# Reproducible MAKD-HAR experiment: WISDM v1.1

This notebook uses subject-disjoint partitions, train-only scaling, five distinct teachers, coalition-based Shapley values, scale-independent teacher weighting, saved per-seed results, and corrected statistical summaries.

Run the quick check first. Never copy quick-check values into the manuscript.


## Corrected data protocol
This notebook uses `wisdm-records-v2`: semicolon record parsing, trailing-comma repair, training-only scaling, and explicit clock handling. Adjacent positive clock gaps over one second or backward clocks end a segment. Duplicate/nonpositive clocks retain samples in file order at nominal 20 Hz. Invalid sensor records end a segment.

Results use `quick-check-v2` and `production-v2`; previous results are preserved. Run sections 1–5 first and review the corrected quick check before starting production. The historical seed-11 configuration-editing cell is no longer needed.


## 1. Prepare Google Drive and the Colab runtime

Place the repository files directly under `MyDrive/makd-har/`:

```text
MyDrive/makd-har/
  run_experiment.py
  makd_har/
  resources/
    WISDM_ar_v1.1_raw.txt
  results/
```

Code and data are copied to Colab temporary storage for execution. Results save directly to Drive. Edit `DRIVE_ROOT` in the next cell if your working project is elsewhere.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

# Persistent locations in Google Drive.
DRIVE_ROOT = Path('/content/drive/MyDrive/makd-har')
DRIVE_PROJECT_ROOT = DRIVE_ROOT  # repository root; resources/results are external inputs/outputs
DRIVE_DATA_FILE = DRIVE_ROOT / 'resources' / 'WISDM_ar_v1.1_raw.txt'
DRIVE_RESULTS_ROOT = DRIVE_ROOT / 'results' / 'wisdm'

# Fast temporary locations in the Colab runtime.
PROJECT_ROOT = Path('/content/revised_pipeline')
DATA_ROOT = Path('/content/WISDM_ar_v1.1_raw.txt')

assert (DRIVE_PROJECT_ROOT / 'run_experiment.py').is_file(), (
    f'Project not found: {DRIVE_PROJECT_ROOT}'
)
assert DRIVE_DATA_FILE.is_file(), f'WISDM raw file not found: {DRIVE_DATA_FILE}'

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
shutil.copytree(
    DRIVE_PROJECT_ROOT,
    PROJECT_ROOT,
    ignore=shutil.ignore_patterns('.git', 'resources', 'results', '__pycache__', '*.pyc'),
)
shutil.copy2(DRIVE_DATA_FILE, DATA_ROOT)

RESULTS_ROOT = DRIVE_RESULTS_ROOT
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

%cd {PROJECT_ROOT}
print(f'Project: {PROJECT_ROOT}')
print(f'Dataset: {DATA_ROOT}')
print(f'Persistent results: {RESULTS_ROOT}')


## 2. Install the declared environment


In [ ]:
%pip install -q -r requirements-colab.txt

# If Colab asks for a runtime restart after installation, restart it and rerun
# Sections 1 and 2 before continuing.


## 3. Verify the runtime and raw dataset


In [ ]:
import sys
import tensorflow as tf

print('Python:', sys.version.split()[0])
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
print('WISDM file size (MB):', round(DATA_ROOT.stat().st_size / (1024 ** 2), 2))
assert (PROJECT_ROOT / 'run_experiment.py').is_file()
assert DATA_ROOT.is_file() and DATA_ROOT.stat().st_size > 0


## 4. Quick structural check

This one-seed run checks WISDM parsing, real 20 Hz window features, subject-disjoint splitting, the five teachers, coalition evaluation, Shapley weighting, and student training. Its values are not manuscript results.


In [ ]:
import subprocess
import sys

quick_command = [
    sys.executable, 'run_experiment.py',
    '--dataset', 'wisdm',
    '--data-root', str(DATA_ROOT),
    '--output', str(RESULTS_ROOT / 'quick-check-v2'),
    '--quick',
    '--verbose', '1',
]
subprocess.run(quick_command, check=True)


## 5. Inspect quick-check artifacts

Confirm that the output contains the dataset manifest and seed result. Check that train, validation, and test user lists are disjoint and that the Shapley diagnostics do not silently conceal uniform weighting.


In [ ]:
quick_files = sorted(
    str(path.relative_to(RESULTS_ROOT))
    for path in (RESULTS_ROOT / 'quick-check-v2').rglob('*')
    if path.is_file()
)
print(*quick_files, sep='\n')
assert quick_files, 'The quick check produced no result files.'


## 6. Validation-only weighting pilot

Run this resumable three-seed pilot before production. It trains and selects candidates using training and validation data only; it does not compute test metrics. It compares 20- and 60-epoch coalition games, raw Shapley weights, 50% shrinkage toward uniform weights, and the uniform control.


In [ ]:
import subprocess

pilot_command = [
    sys.executable, 'run_weighting_pilot.py',
    '--dataset', 'wisdm',
    '--data-root', str(DATA_ROOT),
    '--output', str(RESULTS_ROOT / 'weighting-pilot-v2'),
    '--verbose', '1',
]
subprocess.run(pilot_command, check=True)


## 7. Inspect and share the pilot summary

Do not start production yet. Share the two summary files for review. Production remains blocked until the selected weighting rule is implemented and an approval marker matching that rule is written.


In [ ]:
import json

pilot_root = RESULTS_ROOT / 'weighting-pilot-v2'
pilot_summary = json.loads((pilot_root / 'pilot-summary.json').read_text())
print(json.dumps(pilot_summary, indent=2))
print((pilot_root / 'pilot-summary.csv').read_text())


## 8. Approved production experiment

The predefined validation rule selected raw Shapley weights from the 60-epoch coalition game. The approval record binds this choice to the audited pilot summary. Production uses all ten seeds and remains resumable.


In [ ]:
import json
import subprocess

approval_path = RESULTS_ROOT / 'weighting-pilot-v2' / 'production-approved.json'
assert approval_path.is_file(), 'The audited production approval record is missing.'
approval = json.loads(approval_path.read_text())
assert approval['protocol_version'] == 'records-v2-predict-v2-weight-production-v1'
assert approval['selected_candidate'] == 'shapley-60-raw'
assert approval['coalition_epochs'] == 60
assert approval['uniform_shrinkage'] == 0.0
assert approval['pilot_summary_sha256'] == '48b33c4ce91276d5bf095b0ec49efb59d78d4845db29154d2c09590b4b7da04c'

quick_result_path = RESULTS_ROOT / 'quick-check-v2' / 'seed-11.json'
assert quick_result_path.is_file(), 'Run the corrected quick check first.'
quick_result = json.loads(quick_result_path.read_text())
assert quick_result['config']['protocol_version'] == 'records-v2-predict-v2'
assert quick_result['dataset_manifest']['preprocessing']['protocol'] == 'wisdm-records-v2'
assert quick_result['methods']['svm']['prediction_rule'] == 'SVC.predict for validation and test'

production_command = [
    sys.executable, 'run_experiment.py',
    '--dataset', 'wisdm',
    '--data-root', str(DATA_ROOT),
    '--output', str(RESULTS_ROOT / 'production-v2'),
    '--shapley-mode', 'exact',
    '--teacher-epochs', '60',
    '--student-epochs', '60',
    '--coalition-epochs', '60',
    '--batch-size', '128',
    '--patience', '10',
    '--protocol-version', 'records-v2-predict-v2-weight-production-v1',
    '--selection-record', str(approval_path),
    '--verbose', '1',
]
subprocess.run(production_command, check=True)


## 9. Generate mean ± SD and corrected significance tests


In [ ]:
import subprocess

subprocess.run([
    sys.executable,
    'summarize_results.py',
    str(RESULTS_ROOT / 'production-v2'),
], check=True)


## 10. Create a shareable production-results archive


In [ ]:
archive_base = Path('/content/wisdm-production-v2-results')
local_archive = Path(shutil.make_archive(
    str(archive_base),
    'zip',
    root_dir=RESULTS_ROOT / 'production-v2',
))
drive_archive = RESULTS_ROOT / local_archive.name
shutil.copy2(local_archive, drive_archive)
local_archive.unlink()
print(f'Saved: {drive_archive}')


## Required audit before manuscript use

Confirm ten complete seed files; no NaN values; disjoint user lists in `dataset-manifest.json`; all 32 coalition evaluations per exact-Shapley run; meaningful weighting diagnostics when claiming an adaptive-weighting benefit; and agreement between `summary.csv` and every manuscript table. Energy and MAC values remain absent until measured with validated target-hardware procedures.
